Comparing Efficient Multi-Head Attention Implementations

In [ ]:
import torch

torch.manual_seed(123)  # 固定随机种子，保证权重初始化 / 随机输入在多次运行间可复现
# 按优先级自动选择计算设备：Apple Silicon 的 MPS > NVIDIA 的 CUDA > 最后回退到 CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")   # Apple Silicon GPU (Metal)
elif torch.cuda.is_available():
    device = torch.device("cuda")  # NVIDIA GPU
else:
    device = torch.device("cpu")   # CPU fallback

print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

# 定义本 notebook 中所有多头注意力实现共用的超参数，方便横向对比正确性与速度
batch_size = 8        # 批大小
context_len = 1024    # 序列长度（token 数）
embed_dim = 768       # 嵌入维度（这里 d_in = d_out = 768）
# 构造一个随机输入张量，模拟经过词嵌入+位置嵌入后的输入序列
# 形状：(batch_size, context_len, embed_dim) = (8, 1024, 768)，后面所有实现都喂入这同一份数据
embeddings = torch.randn((batch_size, context_len, embed_dim), device=device)

1. CausalAttention MHA wrapper class from chapter 3

In [ ]:
import torch.nn as nn

# 第3章的单头因果自注意力实现（最基础版本，没有多头概念）
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)  # New
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))  # New

    def forward(self, x):
        # x 形状: (b, num_tokens, d_in)
        b, num_tokens, d_in = x.shape  # New batch dimension b
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # (b, num_tokens, d_out) @ (b, d_out, num_tokens) -> (b, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(1, 2)  # Changed transpose
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)  # New

        context_vec = attn_weights @ values
        return context_vec


# 朴素的多头实现：为每个头创建一个独立的 CausalAttention 子模块
# 效率较低——num_heads 次独立的小矩阵乘法在 Python 循环里串行调用，
# 无法像“单个大矩阵乘法”那样充分利用 GPU 并行度，是本 notebook 中最慢的实现之一
class Ch03_MHA_Wrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )
        # 输出投影层：把 num_heads 个头拼接后的向量（维度 d_out*num_heads）再线性变换一次
        self.out_proj = nn.Linear(d_out*num_heads, d_out*num_heads)

    def forward(self, x):
        # 依次调用每个头（Python for 循环，串行），再在最后一维拼接: (b, num_tokens, d_out*num_heads)
        context_vec = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.out_proj(context_vec)


# 每个头的输出维度设为 embed_dim//12，12 个头拼接后维度正好恢复到 embed_dim
mha_ch03_wrapper = Ch03_MHA_Wrapper(
    d_in=embed_dim,
    d_out=embed_dim//12,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
).to(device)

out = mha_ch03_wrapper(embeddings)
print(out.shape)

2. The multi-head attention class from chapter 3

In [ ]:
# 第3章“高效”版多头注意力：只用一组 Linear 一次性算出完整的 Q/K/V，
# 再通过 view + transpose 把最后一维“隐式拆分”成 (num_heads, head_dim)，
# 所有头的注意力计算都在一次批量矩阵乘法中完成，比上面的 Ch03_MHA_Wrapper 快得多
class Ch03_MHA(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        # 一次性对整个序列做线性变换，得到完整的 d_out 维输出（尚未拆分成多头）
        keys = self.W_key(x)  # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        # (b, num_heads, num_tokens, head_dim) @ (b, num_heads, head_dim, num_tokens)
        # -> (b, num_heads, num_tokens, num_tokens)，每个头独立算注意力分数，但底层是一次批量matmul
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        # contiguous() 是必须的：transpose 之后张量在内存中不再连续，view 需要连续内存
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # optional projection，融合多头信息的最终线性层

        return context_vec


mha_ch03 = Ch03_MHA(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
).to(device)

out = mha_ch03(embeddings)
print(out.shape)

3. An alternative multi-head attention with combined weights

In [ ]:
import torch.nn as nn


# 另一种写法：把 W_query/W_key/W_value 三个 Linear 合并成一个输出 3*d_out 的 Linear，
# 一次矩阵乘法同时算出 Q、K、V（而不是3次独立的矩阵乘法），能更好地利用 GPU，通常比上面的 Ch03_MHA 略快
class MultiHeadAttentionCombinedQKV(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_length, dropout=0.0, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0, "d_out is indivisible by num_heads"

        self.num_heads = num_heads
        self.context_length = context_length
        self.head_dim = d_out // num_heads

        self.qkv = nn.Linear(d_in, 3 * d_out, bias=qkv_bias)
        self.proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        batch_size, num_tokens, embed_dim = x.shape

        # (b, num_tokens, embed_dim) --> (b, num_tokens, 3 * embed_dim)
        # 单个 Linear 同时得到 Q、K、V 三部分拼接后的结果
        qkv = self.qkv(x)

        # (b, num_tokens, 3 * embed_dim) --> (b, num_tokens, 3, num_heads, head_dim)
        qkv = qkv.view(batch_size, num_tokens, 3, self.num_heads, self.head_dim)

        # (b, num_tokens, 3, num_heads, head_dim) --> (3, b, num_heads, num_tokens, head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # (3, b, num_heads, num_tokens, head_dim) -> 3 times (b, num_head, num_tokens, head_dim)
        # unbind(0) 沿第0维拆开，得到 queries, keys, values 三个张量（不产生额外拷贝）
        queries, keys, values = qkv.unbind(0)

        # (b, num_heads, num_tokens, head_dim) --> (b, num_heads, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(-2, -1)
        attn_scores = attn_scores.masked_fill(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # (b, num_heads, num_tokens, num_tokens) --> (b, num_heads, num_tokens, head_dim)
        context_vec = attn_weights @ values

        # (b, num_heads, num_tokens, head_dim) --> (b, num_tokens, num_heads, head_dim)
        context_vec = context_vec.transpose(1, 2)

        # (b, num_tokens, num_heads, head_dim) --> (b, num_tokens, embed_dim)
        context_vec = context_vec.contiguous().view(batch_size, num_tokens, embed_dim)

        context_vec = self.proj(context_vec)

        return context_vec


mha_combined_qkv = MultiHeadAttentionCombinedQKV(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
).to(device)

out = mha_combined_qkv(embeddings)
print(out.shape)

4. Multi-head attention with Einsum

In [ ]:
import math


# 用手写的 nn.Parameter 权重矩阵 + torch.einsum 实现多头注意力，
# 数学上与 Ch03_MHA 等价（同样是单组权重、view+transpose 拆头），
# 只是把 nn.Linear/@ 换成显式的 einsum 表达式，效率通常相近（einsum 底层也会调用 bmm）
class MHAEinsum(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Parameter(torch.randn(d_in, d_out))
        self.W_key = nn.Parameter(torch.randn(d_in, d_out))
        self.W_value = nn.Parameter(torch.randn(d_in, d_out))

        if qkv_bias:
            self.bias_q = nn.Parameter(torch.zeros(d_out))
            self.bias_k = nn.Parameter(torch.zeros(d_out))
            self.bias_v = nn.Parameter(torch.zeros(d_out))
        else:
            self.register_parameter("bias_q", None)
            self.register_parameter("bias_k", None)
            self.register_parameter("bias_v", None)

        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))
        self.reset_parameters()


    # 手动初始化参数，模拟 nn.Linear 默认的 kaiming_uniform_ 初始化方式
    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.W_query, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.W_key, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.W_value, a=math.sqrt(5))
        if self.bias_q is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.W_query)
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.bias_q, -bound, bound)
            nn.init.uniform_(self.bias_k, -bound, bound)
            nn.init.uniform_(self.bias_v, -bound, bound)

    def forward(self, x):
        b, n, _ = x.shape

        # Calculate Q, K, V using einsum, first perform linear transformations
        # "bnd,do->bno": batch(b) x tokens(n) x d_in(d) 与 d_in(d) x d_out(o) 做矩阵乘法 -> (b, n, d_out)
        Q = torch.einsum("bnd,do->bno", x, self.W_query)
        K = torch.einsum("bnd,do->bno", x, self.W_key)
        V = torch.einsum("bnd,do->bno", x, self.W_value)

        # Add biases if they are used
        if self.bias_q is not None:
            Q += self.bias_q
            K += self.bias_k
            V += self.bias_v

        # Reshape for multi-head attention
        Q = Q.view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(b, n, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention
        # "bhnd,bhmd->bhnm": 对每个 batch(b)、每个头(h)，query 位置 n 与 key 位置 m 做点积 -> (b, h, n, m)
        scores = torch.einsum("bhnd,bhmd->bhnm", Q, K) / (self.head_dim ** 0.5)

        # Apply mask
        mask = self.mask[:n, :n]
        scores = scores.masked_fill(mask.bool(), -torch.inf)

        # Softmax and dropout
        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Aggregate the attended context vectors
        # "bhnm,bhmd->bhnd": 注意力权重 (b,h,n,m) 与 V (b,h,m,d) 加权求和 -> (b, h, n, head_dim)
        context_vec = torch.einsum("bhnm,bhmd->bhnd", attn_weights, V)

        # Combine heads and project the output
        context_vec = context_vec.transpose(1, 2).reshape(b, n, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec


mha_einsum = MHAEinsum(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
).to(device)

out = mha_einsum(embeddings)
print(out.shape)

5. Multi-head attention with PyTorch's scaled dot product attention and FlashAttention

In [ ]:
# 使用合并的 QKV 投影（同上面的 MultiHeadAttentionCombinedQKV），
# 但核心注意力计算改用 PyTorch 内置的 F.scaled_dot_product_attention (SDPA)。
# SDPA 会根据硬件/输入自动派发到融合核（如 FlashAttention、memory-efficient attention 等），
# 通常比手写实现快很多、显存占用也更低（尤其是长序列时不需要显式生成 num_tokens x num_tokens 的分数矩阵）
class MHAPyTorchScaledDotProduct(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_length, dropout=0.0, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0, "d_out is indivisible by num_heads"

        self.num_heads = num_heads
        self.context_length = context_length
        self.head_dim = d_out // num_heads
        self.d_out = d_out

        self.qkv = nn.Linear(d_in, 3 * d_out, bias=qkv_bias)
        self.proj = nn.Linear(d_out, d_out)
        self.dropout = dropout

    def forward(self, x):
        batch_size, num_tokens, embed_dim = x.shape

        # (b, num_tokens, embed_dim) --> (b, num_tokens, 3 * embed_dim)
        qkv = self.qkv(x)

        # (b, num_tokens, 3 * embed_dim) --> (b, num_tokens, 3, num_heads, head_dim)
        qkv = qkv.view(batch_size, num_tokens, 3, self.num_heads, self.head_dim)

        # (b, num_tokens, 3, num_heads, head_dim) --> (3, b, num_heads, num_tokens, head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # (3, b, num_heads, num_tokens, head_dim) -> 3 times (b, num_heads, num_tokens, head_dim)
        queries, keys, values = qkv

        use_dropout = 0. if not self.training else self.dropout

        # is_causal=True: 使用内置的因果掩码逻辑，不需要我们自己创建/传入 mask 矩阵，
        # 这样 SDPA 才能安全地派发到 FlashAttention 等融合核（见下面第6节的对比）
        context_vec = nn.functional.scaled_dot_product_attention(
            queries, keys, values, attn_mask=None, dropout_p=use_dropout, is_causal=True)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.transpose(1, 2).contiguous().view(batch_size, num_tokens, self.d_out)

        context_vec = self.proj(context_vec)

        return context_vec

In [ ]:
mha_pytorch_scaled = MHAPyTorchScaledDotProduct(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
).to(device)

out = mha_pytorch_scaled(embeddings)
print(out.shape)

6. PyTorch's scaled dot product attention without FlashAttention

In [ ]:
# 结构与上面第5节的 MHAPyTorchScaledDotProduct 几乎一样，
# 区别在于这里显式创建了一个布尔 mask 并通过 attn_mask 参数传给 SDPA（is_causal=False）。
# 传入自定义 attn_mask 会让 PyTorch 无法派发到高度优化的 FlashAttention 融合核，
# 只能退回到显存效率较低的 memory-efficient / 数学实现路径，因此通常比第5节慢，故名 "without Flash"
class MHAPyTorchSDPAWithoutFlash(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_length, dropout=0.0, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0, "d_out is indivisible by num_heads"

        self.num_heads = num_heads
        self.context_length = context_length
        self.head_dim = d_out // num_heads
        self.d_out = d_out

        self.qkv = nn.Linear(d_in, 3 * d_out, bias=qkv_bias)
        self.proj = nn.Linear(d_out, d_out)
        self.dropout = dropout
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1).bool())

    def forward(self, x):
        batch_size, num_tokens, embed_dim = x.shape

        # (b, num_tokens, embed_dim) --> (b, num_tokens, 3 * embed_dim)
        qkv = self.qkv(x)

        # (b, num_tokens, 3 * embed_dim) --> (b, num_tokens, 3, num_heads, head_dim)
        qkv = qkv.view(batch_size, num_tokens, 3, self.num_heads, self.head_dim)

        # (b, num_tokens, 3, num_heads, head_dim) --> (3, b, num_heads, num_tokens, head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # (3, b, num_heads, num_tokens, head_dim) -> 3 times (b, num_heads, num_tokens, head_dim)
        queries, keys, values = qkv

        use_dropout = 0. if not self.training else self.dropout

        # Ensure attn_mask is compatible with expected shape and `batch_first=True`
        # No need to manually adjust for num_heads; ensure it's right for the sequence
        if self.context_length >= num_tokens:
            attn_mask = self.mask[:num_tokens, :num_tokens]
        else:
            attn_mask = self.mask[:self.context_length, :self.context_length]

        # SDPA uses True for positions that may participate in attention
        # 这里对 mask 取反(~)：self.mask 中 True 表示“需要被屏蔽”的未来位置，
        # 而 SDPA 的 attn_mask 语义是 True 表示“允许参与注意力”的位置，两者刚好相反
        context_vec = nn.functional.scaled_dot_product_attention(
            queries, keys, values, attn_mask=~attn_mask, dropout_p=use_dropout, is_causal=False)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.transpose(1, 2).contiguous().view(batch_size, num_tokens, self.d_out)

        context_vec = self.proj(context_vec)

        return context_vec

In [ ]:
mha_pytorch_sdpa_no_flash = MHAPyTorchSDPAWithoutFlash(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
).to(device)

out = mha_pytorch_sdpa_no_flash(embeddings)
print(out.shape)

7. Using PyTorch's torch.nn.MultiheadAttention

In [ ]:
import torch.nn as nn


# 直接使用 PyTorch 官方的 nn.MultiheadAttention（batch_first=True，输入/输出形状为 (b, seq, embed)）
class MHAPyTorchClass(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_length, dropout=0.0, qkv_bias=False, need_weights=True):
        super().__init__()

        self.context_length = context_length
        # 风险提示：这里把 add_bias_kv 也复用了 qkv_bias 这个开关。
        # bias 控制的是 Q/K/V/输出投影层是否带偏置，而 add_bias_kv 是给 key/value 序列末尾
        # 额外拼接一个可学习的“bias token”（属于不同的机制），两者被强行绑在一起，
        # 如果只是想控制线性层 bias，这里容易被忽略，需要确认是否是有意为之
        self.multihead_attn = nn.MultiheadAttention(
            embed_dim=d_out,
            num_heads=num_heads,
            dropout=dropout,
            bias=qkv_bias,
            add_bias_kv=qkv_bias,
            batch_first=True,
        )

        self.need_weights = need_weights
        self.proj = nn.Linear(d_out, d_out)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1).bool())

    def forward(self, x):
        batch_size, num_tokens, _ = x.shape

        # Ensure attn_mask is compatible with expected shape and `batch_first=True`
        # No need to manually adjust for num_heads; ensure it's right for the sequence
        if self.context_length >= num_tokens:
            attn_mask = self.mask[:num_tokens, :num_tokens]
        else:
            attn_mask = self.mask[:self.context_length, :self.context_length]

        # attn_mask broadcasting will handle batch_size dimension implicitly
        # need_weights=True 时，PyTorch 需要显式计算并返回 softmax 注意力权重矩阵，
        # 这会阻止内部走高效的 SDPA/FlashAttention 融合路径，因此比 need_weights=False 慢（见下面第8节）
        attn_output, _ = self.multihead_attn(
            x, x, x, attn_mask=attn_mask, need_weights=self.need_weights
        )

        output = self.proj(attn_output)

        return output


mha_pytorch_class_default = MHAPyTorchClass(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
).to(device)

out = mha_pytorch_class_default(embeddings)
print(out.shape)

8. Using PyTorch's torch.nn.MultiheadAttention with scaled_dot_product_attention

In [ ]:
mha_pytorch_class_noweights = MHAPyTorchClass(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False,
    need_weights=False # NEW! 关闭注意力权重的返回，PyTorch 内部可以走更高效的融合核路径（更快，但拿不到权重矩阵）
).to(device)

out = mha_pytorch_class_noweights(embeddings)
print(out.shape)

9. Using PyTorch's FlexAttention

In [ ]:
from packaging.version import parse as parse_version

# torch.__version__ 里可能带有非标准后缀（如 "2.5.0+cu121" 或 "2.6.0.dev..."），
# normalize_version 只保留 major.minor.micro，避免这些后缀干扰版本号比较
def normalize_version(version):
    parsed_version = parse_version(version)
    return parse_version(f"{parsed_version.major}.{parsed_version.minor}.{parsed_version.micro}")

current_version = normalize_version(torch.__version__)
MIN_TORCH_VERSION = "2.5.0"  # FlexAttention 需要 PyTorch 2.5.0 及以上版本
required_version = parse_version(MIN_TORCH_VERSION)

In [ ]:
# 只有在 torch 版本满足要求、且当前有可用的 CUDA 设备时才导入 FlexAttention 相关 API
# （FlexAttention 目前主要面向 CUDA 后端）
if current_version >= required_version and torch.cuda.is_available():
    from torch.nn.attention.flex_attention import flex_attention, create_block_mask


# mask_mod 函数：定义“因果”规则，返回 True 表示该 (query 位置, key 位置) 组合允许参与注意力
# 即 key 的位置不能晚于 query 的位置；供 create_block_mask 用来生成稀疏的因果块掩码
def causal(b, h, q_idx, kv_idx):
    return q_idx >= kv_idx


# FlexAttention 是 PyTorch 2.5+ 引入的“可编程注意力”API：
# 通过自定义的 mask_mod / score_mod 函数即可表达因果、滑动窗口、ALiBi 等多种注意力变体，
# 同时仍可编译成高效的融合 kernel，兼顾灵活性与速度
class MHAPyTorchFlexAttention(nn.Module):

    def __init__(self, d_in, d_out, num_heads, context_length, dropout=0.0, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0, "d_out is indivisible by num_heads"

        self.num_heads = num_heads
        self.context_length = context_length
        self.head_dim = d_out // num_heads
        self.d_out = d_out

        self.qkv = nn.Linear(d_in, 3 * d_out, bias=qkv_bias)
        self.proj = nn.Linear(d_out, d_out)
        self.dropout = dropout

        # Since slicing the BlockMask is no longer supported in PyTorch 2.10 and newer
        # we will create a new mask in the forward pass with the correct sequence length
        # 中文说明：早期版本可以在 __init__ 里预先创建好固定长度的 BlockMask，再在 forward 里按实际
        # 序列长度切片使用；但 PyTorch 2.10 起不再支持对 BlockMask 做切片，所以改为在 forward 里
        # 按当前 num_tokens 现场创建 mask（见下面 attn_mask = create_block_mask(...) 那一行）
        # self.block_mask = create_block_mask(causal, B=None, H=None, Q_LEN=context_length, KV_LEN=context_length)


    def forward(self, x):
        batch_size, num_tokens, embed_dim = x.shape

        # (b, num_tokens, embed_dim) --> (b, num_tokens, 3 * embed_dim)
        qkv = self.qkv(x)

        # (b, num_tokens, 3 * embed_dim) --> (b, num_tokens, 3, num_heads, head_dim)
        qkv = qkv.view(batch_size, num_tokens, 3, self.num_heads, self.head_dim)

        # (b, num_tokens, 3, num_heads, head_dim) --> (3, b, num_heads, num_tokens, head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # (3, b, num_heads, num_tokens, head_dim) -> 3 times (b, num_heads, num_tokens, head_dim)
        queries, keys, values = qkv

        # use_dropout = 0. if not self.training else self.dropout

        # Ensure attn_mask is compatible with expected shape and `batch_first=True`
        # No need to manually adjust for num_heads; ensure it's right for the sequence
        # if self.context_length >= num_tokens:
        #     attn_mask = self.block_mask[:num_tokens, :num_tokens]
        # else:
        #     attn_mask = self.block_mask[:self.context_length, :self.context_length]
        #
        #
        # Commented out code lines above since slicing a BlockMask no longer works in PyTorch 3.10
        # Instead, we create a fresh mask each time:
        # 按当前实际序列长度现场生成块掩码（BlockMask 是一种稀疏表示，只存储需要计算的块，省显存）
        attn_mask = create_block_mask(causal, B=None, H=None, Q_LEN=num_tokens, KV_LEN=num_tokens, device=x.device)

        # flex_attention 调用编译后的融合 kernel，根据 block_mask 只计算允许的（因果）块
        context_vec = flex_attention(queries, keys, values, block_mask=attn_mask)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.transpose(1, 2).contiguous().view(batch_size, num_tokens, self.d_out)

        context_vec = self.proj(context_vec)

        return context_vec

In [ ]:
# 同样地，只有版本和 CUDA 条件都满足时才实例化并跑一次 FlexAttention 实现做正确性检查
if current_version >= required_version and torch.cuda.is_available():

    mha_pytorch_flex = MHAPyTorchFlexAttention(
        d_in=embed_dim,
        d_out=embed_dim,
        context_length=context_len,
        dropout=0.0,
        num_heads=12,
        qkv_bias=False
    ).to(device)

    out = mha_pytorch_flex(embeddings)
    print(out.shape)

10. Quick speed comparisons

In [ ]:
# 注意：这里重新选择设备时只判断了 cuda/cpu，没有像最开始 cell 那样考虑 mps，
# 如果在 Apple Silicon 上运行，这里会得到 "cpu" 而不是 "mps"（与最初创建 embeddings 时的设备可能不一致，需留意）
torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Running on {device}")
# 下面用 IPython 的 %timeit 魔法命令做一个快速的（基于 CPU 墙钟时间的）速度对比，
# 对异步执行的 GPU 运算来说精度有限；更精确的 CUDA 事件计时见本notebook第11节


In [ ]:
## 1) CausalAttention MHA wrapper class from chapter 3
# 计时 1）最朴素实现：每个头独立子模块 + Python 循环拼接，预期是这里最慢的实现之一
%timeit mha_ch03_wrapper(embeddings)

In [ ]:
## 2) The multi-head attention class from chapter 3
# 计时 2）单组 QKV 线性层 + view/transpose 拆分多头（第3章“高效”版），比 1）快很多
%timeit mha_ch03(embeddings)

In [ ]:
## 3) An alternative multi-head attention with combined weights
# 计时 3）合并 QKV 为一个 Linear，一次矩阵乘法同时得到 Q/K/V
%timeit mha_combined_qkv(embeddings)

In [ ]:
## 4) Multi-head attention using Einstein summation
# 计时 4）用 einsum 手写实现，数学上与 2）等价
%timeit mha_einsum(embeddings)

In [ ]:
## 5) Multi-head attention with PyTorch's scaled dot product attention
# 计时 5）调用 PyTorch 内置 scaled_dot_product_attention，可能自动使用 FlashAttention 等融合核
%timeit mha_pytorch_scaled(embeddings)

In [ ]:
## 6) PyTorch's scaled dot product attention without FlashAttention
# 计时 6）同样用 SDPA，但显式传入 mask，无法使用 FlashAttention 融合核，预期比 5）慢
%timeit mha_pytorch_sdpa_no_flash(embeddings)

In [ ]:
## 7) Using PyTorch's torch.nn.MultiheadAttention
# 计时 7）官方 nn.MultiheadAttention，need_weights 默认为 True
%timeit mha_pytorch_class_default(embeddings)

In [ ]:
## 8) Using PyTorch's torch.nn.MultiheadAttention disabling `need_weights`
# 计时 8）同 7）但 need_weights=False，可内部走更高效的路径，预期比 7）快
%timeit mha_pytorch_class_noweights(embeddings)

In [ ]:
## 9) Using PyTorch's FlexAttention

# Requires PyTorch 2.5.0 or newer and currently only supports CUDA PyTorch
# 计时 9）FlexAttention，需要 PyTorch >= 2.5 且当前有 CUDA 才会真正执行
%timeit mha_pytorch_flex(embeddings)

10.2 Quick speed comparison on Nvidia A100 GPU

In [ ]:
# Enable tensor cores
# 把 float32 矩阵乘法精度设为 "high"：在支持 TensorFloat32 (TF32) 的 Ampere 及更新架构 GPU 上，
# 矩阵乘法会自动使用 TF32 精度计算，显著提速（代价是牺牲一点数值精度），下面的基准测试会更贴近实际生产配置
torch.set_float32_matmul_precision("high")

In [ ]:
# 与上面 10. 节相同的设置，重跑一遍是为了在“已启用 TF32 tensor core 加速”的状态下
# 在 Nvidia A100 GPU 上重新做一遍同样的 %timeit 对比
torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Running on {device}")

In [ ]:
## 1) CausalAttention MHA wrapper class from chapter 3
# 计时 1）（启用 TF32 后）最朴素实现：每个头独立子模块 + 循环拼接
%timeit mha_ch03_wrapper(embeddings)

In [ ]:
## 2) The multi-head attention class from chapter 3
# 计时 2）（启用 TF32 后）单组 QKV 线性层 + view/transpose 拆分多头
%timeit mha_ch03(embeddings)

In [ ]:
## 3) An alternative multi-head attention with combined weights
# 计时 3）（启用 TF32 后）合并 QKV 为一个 Linear
%timeit mha_combined_qkv(embeddings)

In [ ]:
## 4) Multi-head attention using Einstein summation
# 计时 4）（启用 TF32 后）einsum 手写实现
%timeit mha_einsum(embeddings)

In [ ]:
## 5) Multi-head attention with PyTorch's scaled dot product attention
# 计时 5）（启用 TF32 后）PyTorch 内置 SDPA，可能自动使用 FlashAttention
%timeit mha_pytorch_scaled(embeddings)

In [ ]:
## 6) PyTorch's scaled dot product attention without FlashAttention
# 计时 6）（启用 TF32 后）SDPA 但显式传 mask，无法使用 FlashAttention
%timeit mha_pytorch_sdpa_no_flash(embeddings)

In [ ]:
## 7) Using PyTorch's torch.nn.MultiheadAttention
# 计时 7）（启用 TF32 后）官方 nn.MultiheadAttention，need_weights=True
%timeit mha_pytorch_class_default(embeddings)

In [ ]:
## 8) Using PyTorch's torch.nn.MultiheadAttention disabling `need_weights`
# 计时 8）（启用 TF32 后）同 7）但 need_weights=False
%timeit mha_pytorch_class_noweights(embeddings)

In [ ]:
## 9) Using PyTorch's FlexAttention

# Requires PyTorch 2.5.0 or newer
# 计时 9）（启用 TF32 后）FlexAttention，需要 PyTorch >= 2.5
%timeit mha_pytorch_flex(embeddings)

1. Visualizations


11.1 Visualization utility functions

In [ ]:
# 第11节使用更严谨的、基于 CUDA Event 的计时方式（而不是 %timeit），
# 并会加 warmup 迭代，结果更适合用来画图对比
torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Running on {device}")

In [ ]:
# 把前面所有已经实例化好的多头注意力模块收集到一个字典里：
# 键是便于阅读的名字，值是对应的模型实例，方便下面统一循环做计时和绘图对比
functions = {
    "1) MHA wrapper class": mha_ch03_wrapper,
    "2) MHA Ch03": mha_ch03,
    "3) MHA with combined QKV weights": mha_combined_qkv,
    "4) MHA with Einsum": mha_einsum,
    "5) MHA with PyTorch scaled_dot_product_attention": mha_pytorch_scaled,
    "6) PyTorch's SDPA, no FlashAttention": mha_pytorch_sdpa_no_flash,
    "7) PyTorch MHA class defaults": mha_pytorch_class_default,
    "8) PyTorch MHA with need_weights=False": mha_pytorch_class_noweights
    }

# 只有版本和 CUDA 条件都满足时，才把 FlexAttention 实现也加入对比
if current_version >= required_version and torch.cuda.is_available():
    functions["9) PyTorch's FlexAttention"] =  mha_pytorch_flex

In [ ]:
import matplotlib.pyplot as plt

# Customize further for dark mode aesthetics
# 以下这些 rcParams 用来把图表整体调成深色主题（背景、坐标轴、文字都改成深色/白色），
# 便于在深色背景的文档或网页中展示
plt.rcParams["figure.facecolor"] = "#121212"
plt.rcParams["axes.facecolor"] = "#121212"
plt.rcParams["axes.edgecolor"] = "white"
plt.rcParams["axes.labelcolor"] = "white"
plt.rcParams["text.color"] = "white"
plt.rcParams["xtick.color"] = "white"
plt.rcParams["ytick.color"] = "white"
plt.rcParams["grid.color"] = "#444444"
plt.rcParams["lines.linewidth"] = 2
plt.rcParams["lines.markersize"] = 8

# 绘制各实现的平均执行时间条形图，errorbar 用标准差表示波动范围，
# 并根据最大耗时动态计算 y 轴上限（多留 40% 空间放数值标注），最后保存为 PDF 并展示
def plot_execution_times(functions, execution_means, execution_stds, filename):

    # Create plot
    fig, ax = plt.subplots()
    bars = ax.bar(functions.keys(), execution_means, yerr=execution_stds, capsize=5, error_kw={'ecolor': 'grey'})

    plt.ylabel("Execution time (ms)")
    plt.xticks(rotation=45, ha="right")

    # Calculate new ylim with a margin
    # 用最大执行时间的 1.4 倍作为 y 轴上限，防止顶部的数值标注和柱子重叠/被裁掉
    max_execution_time = max(execution_means)
    upper_ylim = max_execution_time + 0.4 * max_execution_time  # Adding a 40% margin
    plt.ylim(0, upper_ylim)

    # Annotate bars with execution times
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + (0.05 * upper_ylim), round(yval, 2), ha="center", va="bottom")

    plt.tight_layout()
    plt.savefig(filename)
    plt.show()

11.2 Speed comparison (Nvidia A100 GPU) with warmup (forward pass only)

In [ ]:
# CUDA benchmark code shared by Andrei Aksionov
# and based on code from
# https://github.com/cuda-mode/lectures/blob/main/lecture1/pytorch_square.py

import numpy as np

# 用 torch.cuda.Event 对 GPU 运算做精确计时，而不是用 Python 的 time 模块，
# 原因是 CUDA 运算是异步的：CPU 提交 kernel 后会立刻返回，不加同步的话 time 模块量出来的
# 只是“提交耗时”，而不是“GPU 真正执行完的耗时”
def time_pytorch_function(func, *input, num_repeats=1_000):
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    # Warmup
    # 先热身跑5次：消除首次调用时 kernel 编译/显存分配缓存等一次性开销对计时的影响
    for _ in range(5):
        func(*input)
    torch.cuda.synchronize()

    times = []
    for _ in range(num_repeats):
        start.record()
        func(*input)
        end.record()
        torch.cuda.synchronize()  # 等待GPU把这次调用真正执行完，才能读取准确的耗时
        times.append(start.elapsed_time(end))

    return np.mean(times), np.std(times)

In [ ]:
# 对 functions 字典里的每个实现都只测前向传播的耗时（不含反向传播）
execution_stats = [time_pytorch_function(fn, embeddings) for fn in functions.values()]
execution_means = [stat[0] for stat in execution_stats]
execution_stds = [stat[1] for stat in execution_stats]


plot_execution_times(functions, execution_means, execution_stds, filename="1_forward-only.pdf")

11.3 Speed comparison (Nvidia A100 GPU) with warmup (forward and backward pass)

In [ ]:
# 执行一次前向传播，把输出求和当作一个标量 loss，再反向传播算梯度，
# 这样可以同时衡量“前向+反向”的总耗时，更贴近真实训练场景
def forward_backward(func, embeddings):
    if embeddings.grad is not None:
        embeddings.grad.zero_()  # 清空上一次残留的梯度，避免影响本次计时/梯度值

    output = func(embeddings)
    loss = output.sum()
    loss.backward()


# 和上面的 time_pytorch_function 思路一样，只是每次调用换成了 forward_backward（含反向传播）
def time_pytorch_function_forward_backward(func, *input, num_repeats = 1_000):
    # CUDA IS ASYNC so can't use python time module
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    # Warmup
    for _ in range(5):
        forward_backward(func, *input)
    torch.cuda.synchronize()

    times = []
    for _ in range(num_repeats):
        start.record()
        forward_backward(func, *input)
        end.record()
        torch.cuda.synchronize()
        times.append(start.elapsed_time(end))

    return np.mean(times), np.std(times)

In [ ]:
# 对每个实现同时测量前向+反向传播的耗时，绘制对比图
execution_stats = [time_pytorch_function_forward_backward(fn, embeddings) for fn in functions.values()]
execution_means = [stat[0] for stat in execution_stats]
execution_stds = [stat[1] for stat in execution_stats]


plot_execution_times(functions, execution_means, execution_stds, filename="2_forward-and-backward.pdf")

11.4 Speed comparison (Nvidia A100 GPU) with warmup and compilation (forward and backward pass)

In [ ]:
import torch._dynamo
# 遇到 torch.compile 不支持/无法追踪的操作时，优雅降级回退到 eager 模式，而不是直接抛异常中断整个 notebook
torch._dynamo.config.suppress_errors = True

# 用 torch.compile 对每个模型函数做即时编译（算子融合、减少 Python 解释开销等），
# 期望能进一步提速；下面用同样的“前向+反向”计时函数比较编译前后的效果
def prepare_function(fn):
    fn = torch.compile(fn)
    return fn
execution_stats = [time_pytorch_function_forward_backward(prepare_function(fn), embeddings) for fn in functions.values()]  # 对每个函数先编译，再测前向+反向耗时
execution_means = [stat[0] for stat in execution_stats]
execution_stds = [stat[1] for stat in execution_stats]


plot_execution_times(functions, execution_means, execution_stds, filename="3_forward-and-backward-compiled.pdf")